## Used Car Data Preprocessing



In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Day12_Used_Car_Preprocessing_Dataset.csv")

df.head()


,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


In [2]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


Shape: (320, 15)

Data types:
Car_ID                 object
Brand                  object
Year                    int64
Mileage_Km              int64
Engine_CC               int64
Power_BHP             float64
Fuel_Type              object
Transmission           object
City                   object
Seller_Type            object
Condition              object
Previous_Owners         int64
Accidents_Reported      int64
Service_Score           int64
Resale_Price_Lakh     float64
dtype: object

Missing values:
Car_ID                0
Brand                 0
Year                  0
Mileage_Km            0
Engine_CC             0
Power_BHP             0
Fuel_Type             0
Transmission          0
City                  0
Seller_Type           0
Condition             0
Previous_Owners       0
Accidents_Reported    0
Service_Score         0
Resale_Price_Lakh     0
dtype: int64

Duplicate rows: 0


## Separate features and target

`Resale_Price_Lakh` is the target because it is the value we want to predict.

`Car_ID` is removed because it is only an identifier and does not provide useful information for prediction.

In [3]:
X = df.drop(columns=["Resale_Price_Lakh", "Car_ID"])
y = df["Resale_Price_Lakh"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)


Features shape: (320, 13)
Target shape: (320,)


##  Train-test split


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


Training rows: 256
Testing rows: 64


## Outlier detection using IQR



In [5]:
numeric_cols = [
    "Year", "Mileage_Km", "Engine_CC", "Power_BHP",
    "Previous_Owners", "Accidents_Reported", "Service_Score"
]

clip_cols = ["Mileage_Km", "Engine_CC", "Power_BHP"]

X_train_p = X_train.copy()
X_test_p = X_test.copy()

bounds = {}

for col in clip_cols:
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    bounds[col] = (lower, upper)

    X_train_p[col] = X_train_p[col].clip(lower, upper)
    X_test_p[col] = X_test_p[col].clip(lower, upper)

    print(col)
    print("Lower limit:", round(lower, 2))
    print("Upper limit:", round(upper, 2))
    print()


Mileage_Km
Lower limit: -31351.75
Upper limit: 175124.25

Engine_CC
Lower limit: 66.0
Upper limit: 2592.0

Power_BHP
Lower limit: 67.46
Upper limit: 233.36



## Encoding categorical variables



In [6]:
from sklearn.preprocessing import OneHotEncoder

condition_map = {
    "Poor": 1,
    "Fair": 2,
    "Good": 3,
    "Very Good": 4,
    "Excellent": 5
}

X_train_p["Condition"] = X_train_p["Condition"].map(condition_map)
X_test_p["Condition"] = X_test_p["Condition"].map(condition_map)

onehot_cols = ["Brand", "Fuel_Type", "Transmission", "City", "Seller_Type"]

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

train_ohe = ohe.fit_transform(X_train_p[onehot_cols])
test_ohe = ohe.transform(X_test_p[onehot_cols])

onehot_names = ohe.get_feature_names_out(onehot_cols)

print("One-hot columns created:", len(onehot_names))


One-hot columns created: 29


## Feature scaling



In [7]:
from sklearn.preprocessing import StandardScaler

scale_cols = numeric_cols + ["Condition"]

scaler = StandardScaler()

train_scaled = scaler.fit_transform(X_train_p[scale_cols])
test_scaled = scaler.transform(X_test_p[scale_cols])

train_processed = pd.DataFrame(
    train_scaled,
    index=X_train.index,
    columns=scale_cols
)

train_processed = pd.concat([
    train_processed,
    pd.DataFrame(
        train_ohe,
        index=X_train.index,
        columns=onehot_names
    )
], axis=1)

train_processed["Resale_Price_Lakh"] = y_train

test_processed = pd.DataFrame(
    test_scaled,
    index=X_test.index,
    columns=scale_cols
)

test_processed = pd.concat([
    test_processed,
    pd.DataFrame(
        test_ohe,
        index=X_test.index,
        columns=onehot_names
    )
], axis=1)

test_processed["Resale_Price_Lakh"] = y_test


## Final processed dataset


In [8]:
train_processed["Split"] = "Train"
test_processed["Split"] = "Test"

processed = pd.concat([
    train_processed,
    test_processed
]).sort_index()

print("Final shape:", processed.shape)
processed.head()


Final shape: (320, 39)


,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Condition,Brand_Honda,Brand_Hyundai,...,City_Jaipur,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Certified Dealer,Seller_Type_Dealer,Seller_Type_Individual,Resale_Price_Lakh,Split
0,0.422486,-0.102145,-0.402934,-0.669911,-0.734379,-0.442634,-0.373821,-0.341927,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.38,Train
1,0.119527,0.439757,-0.956625,-0.113562,-0.734379,-0.442634,0.830435,-0.341927,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.83,Train
2,0.422486,-0.838755,0.250822,1.124864,0.433329,-0.442634,1.071286,0.675903,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,7.30,Train
3,-0.183432,-0.069952,1.636162,-0.041269,1.601038,-0.442634,-0.855524,1.693733,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.82,Test
4,-1.092309,0.788730,0.720014,1.756650,0.433329,-0.442634,0.589584,0.675903,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.93,Train


In [9]:
print("Missing values after preprocessing:")
print(processed.isnull().sum().sum())

print("\nFinal data types:")
print(processed.dtypes.value_counts())

print("\nSplit:")
print(processed["Split"].value_counts())

print("\nFinal shape:", processed.shape)


Missing values after preprocessing:
0

Final data types:
float64    38
object      1
Name: count, dtype: int64

Split:
Split
Train    256
Test      64
Name: count, dtype: int64

Final shape: (320, 39)


In [10]:
processed.to_csv(
    "Day12_Used_Car_Preprocessed_Dataset.csv",
    index=False
)

print("Saved successfully.")


Saved successfully.


## Conclusion

The dataset was split into training and testing data first. Selected numerical outliers were handled using training-set IQR limits, `Condition` was ordinal encoded, nominal categorical columns were one-hot encoded, and numerical features were standardized.

The transformations were fitted only on the training data, so the preprocessing avoids data leakage from the test set.